# 01 - Build the Monthly Palm Oil Research Dataset

This notebook downloads public data, converts each source to monthly frequency, and exports the results as CSV files, which includes source-level CSVs plus one merged CSV. 

Before running the notebook, install `requirements.txt`. The Earth Engine section includes an interactive browser-based authorization step.

## Output files

- `data/interim/mpob_supply_monthly.csv`: Malaysian production, stocks, exports, and imports
- `data/interim/mpob_ffb_yield_monthly.csv`: MPOB monthly FFB yield
- `data/interim/world_bank_prices_monthly.csv`: international palm oil, soybean oil, and crude oil prices
- `data/interim/usd_myr_monthly.csv`: U.S. dollar to Malaysian ringgit exchange rate
- `data/interim/noaa_roni_monthly.csv`: El Nino and La Nina conditions
- `data/interim/sabah_weather_monthly.csv`: rainfall and surface soil moisture in Sabah
- `data/processed/palm_oil_monthly_panel.csv`: monthly merged research panel
- `data/metadata/sources.csv`: data providers and source links
- `data/metadata/data_coverage.csv`: non-null counts and date coverage for every panel column

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.parse import urljoin
import re

import ee
import numpy as np
import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

START_YEAR = 2014
END_YEAR = pd.Timestamp.today().year

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
METADATA_DIR = PROJECT_ROOT / 'data' / 'metadata'
for folder in (INTERIM_DIR, PROCESSED_DIR, METADATA_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SESSION = requests.Session()
SESSION.headers['User-Agent'] = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)

In [ ]:
def number(text):
    cleaned = re.sub(r'[^0-9.\-]', '', str(text))
    return float(cleaned) if cleaned not in {'', '-', '.', '-.'} else np.nan


def save_csv(frame, path):
    frame = frame.sort_values('month').reset_index(drop=True)
    frame.to_csv(path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    print(f'{path.relative_to(PROJECT_ROOT)}: {len(frame)} rows')
    return frame

## 1. MPOB Monthly Supply Fundamentals

This section extracts production, closing stocks, exports, and imports from the Malaysian Palm Oil Board annual monthly tables. All quantities are measured in tonnes. Each MPOB table includes the previous December as a comparison column; the code removes that column and retains January through December of the selected year.

In [ ]:
MPOB_SUMMARY_URL = 'https://bepi.mpob.gov.my/stat/web_internal.php?val={year}84'
SECTION_ROWS = {
    'PRODUCTION': ('Crude Palm Oil', 'cpo_production_t'),
    'CLOSING STOCK': ('Palm Oil', 'closing_stock_t'),
    'EXPORT': ('Palm Oil', 'palm_oil_exports_t'),
    'IMPORT': ('Palm Oil (CPO+PPO)', 'palm_oil_imports_t'),
}


def read_mpob_summary(year):
    response = SESSION.get(MPOB_SUMMARY_URL.format(year=year), timeout=60)
    response.raise_for_status()
    rows = [[cell.get_text(' ', strip=True) for cell in tr.find_all(['th', 'td'])]
            for tr in BeautifulSoup(response.text, 'lxml').find_all('tr')]

    section = None
    found = {}
    for cells in rows:
        if not cells:
            continue
        label = cells[0].strip()
        for heading in SECTION_ROWS:
            if label.upper().startswith(heading):
                section = heading
        if section not in SECTION_ROWS:
            continue
        wanted_label, column = SECTION_ROWS[section]
        if label.casefold() == wanted_label.casefold():
            values = [number(value) for value in cells[1:]]
            found[column] = values[1:13]  # remove previous December

    months = pd.date_range(f'{year}-01-01', periods=12, freq='MS')
    result = pd.DataFrame({'month': months})
    for column in SECTION_ROWS.values():
        name = column[1]
        result[name] = found[name]
    return result


mpob_supply = pd.concat(
    [read_mpob_summary(year) for year in range(START_YEAR, END_YEAR + 1)],
    ignore_index=True,
)
mpob_supply = save_csv(mpob_supply, INTERIM_DIR / 'mpob_supply_monthly.csv')
mpob_supply.tail()

## 2. MPOB Monthly FFB Yield

The 2015 archive is split into two half-year PDFs. Later year-end PDFs contain the current and previous year, providing monthly coverage through 2025. The output retains the Sabah, Sabah/Sarawak, and Malaysia rows available in the reports.

In [ ]:
MPOB_YIELD_ARCHIVE = 'https://bepi.mpob.gov.my/index.php/import/735-archive-yield-list'
MPOB_YIELD_PDF_PARTS = {
    'https://bepi.mpob.gov.my/images/Yield/Yield-2015/FFB%20Yield_January-June_2015.pdf': [(0, range(1, 7))],
    'https://bepi.mpob.gov.my/images/Yield/Yield-2015/FFB_Yield_July-December_2015.pdf': [(0, range(7, 13))],
    'https://bepi.mpob.gov.my/images/Yield/Yield-2017/FFB_Yield_January_Dec_2017.pdf': [(0, range(1, 7)), (1, range(7, 13))],
    'https://bepi.mpob.gov.my/images/Yield/Yield-2019/FFB_Yield_January_Dec_2019.pdf': [(0, range(1, 7)), (1, range(7, 13))],
    'https://bepi.mpob.gov.my/images/Yield/Yield-2021/FFB_Yield_Dec_2021.pdf': [(0, range(1, 7)), (1, range(7, 13))],
    'https://bepi.mpob.gov.my/images/Yield/Yield-2023/FFB_Yield_Dec_2023.pdf': [(0, range(1, 7)), (1, range(7, 13))],
    'https://bepi.mpob.gov.my/images/Yield/Yield-2025/Homepage%20FFB%20Yield%2012%20(Jan-Dec).pdf': [(0, range(1, 7)), (1, range(7, 13))],
}
REGIONS = ['Sabah', 'Sabah & Sarawak', 'Sabah/Sarawak', 'MALAYSIA']


def read_yield_pdf(pdf_url, page_specs):
    response = SESSION.get(pdf_url, timeout=60)
    response.raise_for_status()
    content = response.content
    records = []
    with pdfplumber.open(BytesIO(content)) as pdf:
        for page_number, months in page_specs:
            text = pdf.pages[page_number].extract_text()
            years = list(dict.fromkeys(map(int, re.findall(r'20\d{2}', text[:1200]))))[:2]
            for region in REGIONS:
                match = re.search(rf'(?mi)^{re.escape(region)}\s+(.+)$', text)
                if not match:
                    continue
                values = list(map(float, re.findall(r'\d+\.\d+', match.group(1))))[:12]
                for index, month_number in enumerate(months):
                    for offset, year in enumerate(years):
                        records.append({
                            'month': pd.Timestamp(year, month_number, 1),
                            'region': region.replace('Sabah/Sarawak', 'Sabah & Sarawak'),
                            'ffb_yield_t_ha': values[index * 2 + offset],
                        })
    return pd.DataFrame(records)


ffb_yield = pd.concat(
    [read_yield_pdf(url, page_specs) for url, page_specs in MPOB_YIELD_PDF_PARTS.items()],
    ignore_index=True,
)
ffb_yield = ffb_yield.drop_duplicates(['month', 'region'], keep='last')
ffb_yield = save_csv(ffb_yield, INTERIM_DIR / 'mpob_ffb_yield_monthly.csv')
ffb_yield.tail()

## 3. World Bank International Commodity Prices

This section finds the latest monthly Pink Sheet workbook from the World Bank Commodity Markets page.

In [ ]:
WORLD_BANK_PAGE = 'https://www.worldbank.org/en/research/commodity-markets'


def read_world_bank_prices():
    page = BeautifulSoup(SESSION.get(WORLD_BANK_PAGE, timeout=60).text, 'lxml')
    link = next(a['href'] for a in page.find_all('a', href=True)
                if 'CMO-Historical-Data-Monthly.xlsx' in a['href'])
    data = pd.read_excel(urljoin(WORLD_BANK_PAGE, link), sheet_name='Monthly Prices', skiprows=4)
    data = data.rename(columns={data.columns[0]: 'period'})
    wanted = {
        'Palm oil': 'palm_oil_usd_t',
        'Soybean oil': 'soybean_oil_usd_t',
        'Crude oil, average': 'crude_oil_usd_bbl',
    }
    data = data[['period', *wanted]].rename(columns=wanted)
    data['month'] = pd.to_datetime(data['period'].astype(str), format='%YM%m')
    data = data.drop(columns='period')
    return data[data['month'].dt.year >= START_YEAR]


world_bank = save_csv(read_world_bank_prices(), INTERIM_DIR / 'world_bank_prices_monthly.csv')
world_bank.tail()

## 4. FRED USD/MYR Exchange Rate

The daily FRED DEXMAUS series is converted to a monthly average. The value represents the number of Malaysian ringgit per U.S. dollar.

In [ ]:
FRED_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXMAUS'
fx = pd.read_csv(FRED_URL)
fx = fx.rename(columns={fx.columns[0]: 'date', 'DEXMAUS': 'usd_myr'})
fx['date'] = pd.to_datetime(fx['date'])
fx['usd_myr'] = pd.to_numeric(fx['usd_myr'], errors='coerce')
fx = (fx.set_index('date')['usd_myr'].resample('MS').mean().rename_axis('month').reset_index())
fx = fx[fx['month'].dt.year >= START_YEAR]
fx = save_csv(fx, INTERIM_DIR / 'usd_myr_monthly.csv')
fx.tail()

## 5. NOAA RONI

RONI is NOAA's three-month running Relative Nino 3.4 Index. This section reads NOAA's official machine-readable text file. Each overlapping season, such as DJF or JFM, is assigned to its middle month to create a monthly series.

In [ ]:
RONI_URL = 'https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt'
SEASONS = ['DJF', 'JFM', 'FMA', 'MAM', 'AMJ', 'MJJ', 'JJA', 'JAS', 'ASO', 'SON', 'OND', 'NDJ']


def read_roni():
    response = SESSION.get(RONI_URL, timeout=60)
    response.raise_for_status()
    data = pd.read_csv(StringIO(response.text), sep=r'\s+')
    data = data.rename(columns={'SEAS': 'season', 'YR': 'year', 'ANOM': 'roni'})
    data['month_number'] = data['season'].map({season: i for i, season in enumerate(SEASONS, 1)})
    data['month'] = pd.to_datetime(dict(year=data['year'], month=data['month_number'], day=1))
    return data.loc[data['year'] >= START_YEAR, ['month', 'roni']]


roni = save_csv(read_roni(), INTERIM_DIR / 'noaa_roni_monthly.csv')
roni.tail()

## 6. Google Earth Engine: Sabah Weather

- CHIRPS: monthly accumulated rainfall in millimetres
- ERA5-Land: volumetric soil water in the 0-7 cm surface layer, measured in m3/m3

To keep the two weather series aligned, the end date is set three months before the current month. 

In [ ]:
ee.Authenticate(auth_mode='notebook')
ee.Initialize()
print('Earth Engine authorization completed.')

In [ ]:
sabah = (ee.FeatureCollection('FAO/GAUL/2015/level1')
         .filter(ee.Filter.eq('ADM0_NAME', 'Malaysia'))
         .filter(ee.Filter.eq('ADM1_NAME', 'Sabah'))
         .geometry())
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').select('volumetric_soil_water_layer_1')

weather_end = (pd.Timestamp.today().to_period('M') - 3).to_timestamp()
weather_months = pd.date_range(f'{START_YEAR}-01-01', weather_end, freq='MS')

features = []
for month in weather_months:
    start = ee.Date(month.strftime('%Y-%m-%d'))
    end = start.advance(1, 'month')
    rain = chirps.filterDate(start, end).sum().reduceRegion(
        reducer=ee.Reducer.mean(), geometry=sabah, scale=5500, bestEffort=True
    ).get('precipitation')
    soil = era5.filterDate(start, end).mean().reduceRegion(
        reducer=ee.Reducer.mean(), geometry=sabah, scale=11000, bestEffort=True
    ).get('volumetric_soil_water_layer_1')
    features.append(ee.Feature(None, {
        'month': month.strftime('%Y-%m-%d'),
        'rainfall_mm': rain,
        'soil_moisture_m3_m3': soil,
    }))

weather_info = ee.FeatureCollection(features).getInfo()['features']
weather = pd.DataFrame([feature['properties'] for feature in weather_info])
weather['month'] = pd.to_datetime(weather['month'])
weather = save_csv(weather, INTERIM_DIR / 'sabah_weather_monthly.csv')
weather.tail()

## 7. Merge and Export the Main Panel


In [ ]:
yield_wide = (ffb_yield.pivot(index='month', columns='region', values='ffb_yield_t_ha')
              .rename(columns={
                  'Sabah': 'ffb_yield_sabah_t_ha',
                  'Sabah & Sarawak': 'ffb_yield_sabah_sarawak_t_ha',
                  'MALAYSIA': 'ffb_yield_malaysia_t_ha',
              })
              .reset_index())

calendar = pd.DataFrame({'month': pd.date_range(
    f'{START_YEAR}-01-01', pd.Timestamp.today().to_period('M').to_timestamp(), freq='MS'
)})

panel = calendar
for frame in (mpob_supply, yield_wide, world_bank, fx, roni, weather):
    panel = panel.merge(frame, on='month', how='left')

panel = save_csv(panel, PROCESSED_DIR / 'palm_oil_monthly_panel.csv')
panel.tail(12)

## 8. Export Metadata and Check Coverage


In [ ]:
sources = pd.DataFrame([
    {'dataset': 'MPOB monthly supply', 'provider': 'Malaysian Palm Oil Board', 'url': MPOB_SUMMARY_URL.format(year='YYYY')},
    {'dataset': 'MPOB FFB yield', 'provider': 'Malaysian Palm Oil Board', 'url': MPOB_YIELD_ARCHIVE},
    {'dataset': 'Commodity prices', 'provider': 'World Bank', 'url': WORLD_BANK_PAGE},
    {'dataset': 'USD/MYR', 'provider': 'Federal Reserve Bank of St. Louis', 'url': FRED_URL},
    {'dataset': 'RONI', 'provider': 'NOAA Climate Prediction Center', 'url': RONI_URL},
    {'dataset': 'CHIRPS rainfall', 'provider': 'UCSB CHG via Google Earth Engine', 'url': 'https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY'},
    {'dataset': 'ERA5-Land soil moisture', 'provider': 'ECMWF via Google Earth Engine', 'url': 'https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR'},
])
sources.to_csv(METADATA_DIR / 'sources.csv', index=False, encoding='utf-8-sig')
sources

In [ ]:
coverage = panel.set_index('month').notna().sum().rename('non_null_months').to_frame()
coverage['first_month'] = [panel.loc[panel[column].notna(), 'month'].min() for column in coverage.index]
coverage['last_month'] = [panel.loc[panel[column].notna(), 'month'].max() for column in coverage.index]
coverage.to_csv(METADATA_DIR / 'data_coverage.csv', encoding='utf-8-sig')
coverage